# Estimación de la Función de Desempeño de Seguridad (FDS) para siniestralidad peatonal en Bogotá
### Modelos de conteo (Poisson / Binomial Negativa / ZINB) a nivel de segmento vial, con validación cruzada aleatoria de K = 5 pliegues

**Autor:** Brayan Hernandez - Santiago Toro — Tesis de grado, Ciencia de Datos

## 1. Entorno y reproducibilidad

In [1]:
## Librerías
import sys
import warnings
import numpy as np
import pandas as pd
import statsmodels
import sklearn
import statsmodels.api as sm
from scipy import stats
from scipy.special import expit
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error, mean_squared_error
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP

pd.set_option('display.max_columns', 80)

In [2]:
## Semilla y versiones del entorno (se documentan en la metodología)
SEMILLA = 1949
K_PLIEGUES = 5

print(f"Python: {sys.version.split()[0]}")
print(f"statsmodels: {statsmodels.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")

Python: 3.13.7
statsmodels: 0.14.6
scikit-learn: 1.9.0
pandas: 3.0.5


## 2. Datos

In [3]:
## Carga del conjunto unificado; el índice queda 0..n-1 porque las particiones trabajan por posición
df_unificado = pd.read_csv('Data/processed/segmentos_unificado.csv').reset_index(drop=True)

In [4]:
## Verificación de columnas y de la variable respuesta
COL_DIST = ['dist_puente_peatonal_m', 'dist_puente_vehicular_m', 'dist_semaforo_m', 'ratio_cobertura_peatonal']
COLS_REQ = COL_DIST + ['MVINUMC', 'MVI_VELREG', 'long_seg_m', 'n_siniestros']

faltan = [c for c in COLS_REQ if c not in df_unificado.columns]
assert not faltan, f'Faltan columnas: {faltan}'
assert df_unificado[COLS_REQ].notna().all().all(), 'Hay valores faltantes en las columnas del modelo'
assert (df_unificado['long_seg_m'] > 0).all(), 'Hay segmentos con longitud no positiva'

print(f"Segmentos: {len(df_unificado)} | % con cero siniestros: {100 * (df_unificado['n_siniestros'] == 0).mean():.2f}")
print(df_unificado['n_siniestros'].value_counts().sort_index().head(8).to_string())

Segmentos: 131704 | % con cero siniestros: 89.88
n_siniestros
0    118382
1      9826
2      2062
3       729
4       332
5       147
6        85
7        41


## 3. Offset y covariables

In [5]:
## Respuesta y offset: el logaritmo de la longitud es el único término de exposición
Y_ALL = df_unificado['n_siniestros'].to_numpy(float)
OFF_ALL = np.log(df_unificado['long_seg_m'].to_numpy(float))

## Covariables: log1p en distancias (se estandarizan por pliegue) y dummies de velocidad reglamentaria
X0 = df_unificado[COL_DIST + ['MVINUMC']].copy()
X0[COL_DIST] = np.log1p(X0[COL_DIST])
DUM = pd.get_dummies(df_unificado['MVI_VELREG'].astype(int).astype(str), prefix='vel', drop_first=True, dtype=float)


def armar_disenio(pos_tr, pos_te):
    ## Matriz del conteo (X) y de la inflación (Xi = constante + log-longitud estandarizado); el escalador usa solo entrenamiento
    esc = StandardScaler().fit(X0.iloc[pos_tr][COL_DIST])
    mu_l, sd_l = OFF_ALL[pos_tr].mean(), OFF_ALL[pos_tr].std()

    def _X(pos):
        X = X0.iloc[pos].copy()
        X[COL_DIST] = esc.transform(X[COL_DIST])
        return sm.add_constant(pd.concat([X, DUM.iloc[pos]], axis=1), has_constant='add')

    def _Xi(pos, X):
        return pd.DataFrame({'const': 1.0, 'log_long_z': (OFF_ALL[pos] - mu_l) / sd_l}, index=X.index)

    X_tr, X_te = _X(pos_tr), _X(pos_te)
    return dict(X_tr=X_tr, X_te=X_te, Xi_tr=_Xi(pos_tr, X_tr), Xi_te=_Xi(pos_te, X_te),
                off_tr=OFF_ALL[pos_tr], off_te=OFF_ALL[pos_te], y_tr=Y_ALL[pos_tr], y_te=Y_ALL[pos_te])

## 4. Modelos candidatos

In [6]:
## Ajuste robusto: prueba optimizadores en cadena con arranque en caliente; maxiter explícito (el defecto de statsmodels, 35, no alcanza)
def _num(a):
    return np.asarray(a, dtype=float)


def _fit_robusto(modelo, start=None, plan=(('bfgs', 500), ('lbfgs', 1000), ('nm', 4000))):
    mejor, ok = None, False
    for metodo, it in plan:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            try:
                r = modelo.fit(start_params=start, method=metodo, maxiter=it, disp=0)
            except Exception:
                continue
        p = _num(r.params)
        if not np.all(np.isfinite(p)):
            continue
        conv = bool(r.mle_retvals.get('converged', False))
        if mejor is None or conv or r.llf > mejor.llf:
            mejor, ok = r, conv
        if conv:
            break
        start = p
    if mejor is None:
        raise RuntimeError('Ningún optimizador devolvió parámetros finitos')
    return mejor, ok

In [7]:
## Los cuatro modelos con la misma interfaz: media, logpmf, parámetros, AIC, BIC y convergencia
def ajustar(tipo, y, X, off, Xi=None, params_bn=None):
    y, off, res, alpha, pi = _num(y), _num(off), None, np.nan, None

    if tipo == 'nulo':
        b0 = np.log(y.sum() / np.exp(off).sum())    # MLE cerrado de Poisson solo con intercepto y offset
        media = lambda X, off, Xi=None: np.exp(b0 + _num(off))
        logpmf = lambda y, X, off, Xi=None: stats.poisson.logpmf(_num(y), media(X, off))
        params, ok = np.array([b0]), True

    elif tipo == 'poisson':
        res = sm.GLM(y, X, family=sm.families.Poisson(), offset=off).fit()
        params, ok = _num(res.params), bool(res.converged)
        media = lambda X, off, Xi=None: np.exp(np.clip(_num(X) @ params + _num(off), -30, 30))
        logpmf = lambda y, X, off, Xi=None: stats.poisson.logpmf(_num(y), media(X, off))

    elif tipo == 'bn':
        res, ok = _fit_robusto(sm.NegativeBinomial(y, X, offset=off, loglike_method='nb2'))
        params = _num(res.params)
        b, alpha = params[:-1], max(params[-1], 1e-8)
        if params[-1] < 1e-4:
            warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)
        media = lambda X, off, Xi=None: np.exp(np.clip(_num(X) @ b + _num(off), -30, 30))
        # NB2: n = 1/alpha y p = 1/(1 + alpha*mu) dan media mu y varianza mu + alpha*mu^2
        logpmf = lambda y, X, off, Xi=None: stats.nbinom.logpmf(_num(y), 1 / alpha, 1 / (1 + alpha * media(X, off)))

    else:
        k_inf = Xi.shape[1]
        mod = ZeroInflatedNegativeBinomialP(y, X, exog_infl=Xi, offset=off, inflation='logit', p=2)
        start = np.r_[-3.0, np.zeros(k_inf - 1), _num(params_bn)] if params_bn is not None else None
        res, ok = _fit_robusto(mod, start)
        params = _num(res.params)                      # orden de statsmodels: [inflación, conteo, alpha]
        g, b, alpha = params[:k_inf], params[k_inf:-1], max(params[-1], 1e-8)
        mu = lambda X, off: np.exp(np.clip(_num(X) @ b + _num(off), -30, 30))
        pi = lambda Xi: expit(_num(Xi) @ g)
        media = lambda X, off, Xi: (1 - pi(Xi)) * mu(X, off)

        def logpmf(y, X, off, Xi):
            y_, m_, p_ = _num(y), mu(X, off), pi(Xi)
            lnb = stats.nbinom.logpmf(y_, 1 / alpha, 1 / (1 + alpha * m_))
            with np.errstate(divide='ignore'):
                return np.where(y_ == 0, np.logaddexp(np.log(p_), np.log1p(-p_) + lnb), np.log1p(-p_) + lnb)

    n, k = len(y), len(params)
    llf = float(logpmf(y, X, off, Xi).sum())
    if res is not None and abs(res.llf - llf) > 1e-3 * abs(res.llf) + 1e-2:
        warnings.warn(f'{tipo}: llf propio ({llf:.2f}) != llf de statsmodels ({res.llf:.2f})', RuntimeWarning)
    return dict(tipo=tipo, res=res, media=media, logpmf=logpmf, params=params, k=k, llf=llf,
                aic=-2 * llf + 2 * k, bic=-2 * llf + k * np.log(n), convergio=ok, alpha=alpha, pi=pi)


def ajustar_todos(D):
    ## El ZINB arranca desde los parámetros de la BN
    args = (D['y_tr'], D['X_tr'], D['off_tr'])
    m = {'Nulo': ajustar('nulo', *args), 'Poisson': ajustar('poisson', *args), 'BN': ajustar('bn', *args)}
    m['ZINB'] = ajustar('zinb', *args, D['Xi_tr'], params_bn=m['BN']['params'])
    return m


def metricas(m, y, X, off, Xi=None):
    y = _num(y)
    mu = np.clip(m['media'](X, off, Xi), 1e-9, None)
    return dict(MAE=mean_absolute_error(y, mu), RMSE=float(np.sqrt(mean_squared_error(y, mu))),
                devianza=mean_poisson_deviance(y, mu), logscore=float(m['logpmf'](y, X, off, Xi).mean()))

## 5. Ajuste sobre todos los segmentos y selección entre Poisson y BN

In [8]:
## Ajuste con todos los segmentos; el ZINB usa inflación logit(pi) = g0 + g1*log-longitud estandarizada
pos_all = np.arange(len(df_unificado))
D = armar_disenio(pos_all, pos_all)
aj = ajustar_todos(D)

comp = pd.DataFrame([dict(modelo=nom, parametros=m['k'], logLik=m['llf'], AIC=m['aic'], BIC=m['bic'], alpha=m['alpha'],
                          pi_medio=float(m['pi'](D['Xi_tr']).mean()) if m['pi'] else np.nan, convergio=m['convergio'])
                     for nom, m in aj.items()]).set_index('modelo')
print(comp.round(4).to_string())

C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


         parametros      logLik          AIC          BIC   alpha  pi_medio  convergio
modelo                                                                                
Nulo              1 -63974.3728  127950.7455  127960.5338     NaN       NaN       True
Poisson           7 -55954.4910  111922.9820  111991.5002     NaN       NaN       True
BN                8 -55954.4708  111924.9416  112003.2481  0.0000       NaN       True
ZINB             10 -50271.9868  100563.9737  100661.8568  2.2533    0.1326       True


In [9]:
## Poisson vs BN: razón de verosimilitud sobre alpha con p-valor corregido por la frontera (alpha = 0)
LR = max(2 * (aj['BN']['llf'] - aj['Poisson']['llf']), 0)
P_LR = 0.5 * stats.chi2.sf(LR, 1)    # mezcla 50:50 de chi2(0) y chi2(1), por eso la mitad del p-valor convencional
BASE = 'BN' if P_LR < 0.05 else 'Poisson'

print(f"LR = {LR:.2f} | p (corregido por frontera) = {P_LR:.3g} | alpha BN = {aj['BN']['alpha']:.4f}")
print(f"Modelo base: {BASE}")

LR = 0.04 | p (corregido por frontera) = 0.42 | alpha BN = 0.0000
Modelo base: Poisson


## 6. Validación cruzada aleatoria (K = 5)

In [10]:
## Cinco particiones 80/20 con validaciones disjuntas; cada pliegue reajusta los cuatro modelos
filas = []
for f, (pos_tr, pos_te) in enumerate(KFold(K_PLIEGUES, shuffle=True, random_state=SEMILLA).split(pos_all)):
    Df = armar_disenio(pos_tr, pos_te)
    mods = ajustar_todos(Df)
    dev_nulo = metricas(mods['Nulo'], Df['y_te'], Df['X_te'], Df['off_te'])['devianza']
    for nom, m in mods.items():
        r = metricas(m, Df['y_te'], Df['X_te'], Df['off_te'], Df['Xi_te'])
        filas.append(dict(r, fold=f, modelo=nom, convergio=m['convergio'], dev_explicada=1 - r['devianza'] / dev_nulo))
    print(f'Pliegue {f + 1}/{K_PLIEGUES} listo')

res_cv = pd.DataFrame(filas)

C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


Pliegue 1/5 listo


C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


Pliegue 2/5 listo


C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


Pliegue 3/5 listo


C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


Pliegue 4/5 listo


C:\Users\santiago\AppData\Local\Temp\ipykernel_9096\2564834047.py:22: RuntimeWarning: BN: alpha ~ 0, el ajuste colapsó a Poisson
  warnings.warn('BN: alpha ~ 0, el ajuste colapsó a Poisson', RuntimeWarning)


Pliegue 5/5 listo


In [11]:
## Media y desviación estándar entre pliegues (menor es mejor: MAE, RMSE, devianza; mayor es mejor: logscore, dev_explicada)
metr = ['MAE', 'RMSE', 'devianza', 'logscore', 'dev_explicada']
tabla_cv = res_cv.groupby('modelo')[metr].agg(['mean', 'std']).reindex(['Nulo', 'Poisson', 'BN', 'ZINB'])
print(tabla_cv.round(4).to_string())
print('Ajustes que no convergieron en la validación:', int((~res_cv['convergio']).sum()))

            MAE            RMSE         devianza         logscore         dev_explicada        
           mean     std    mean     std     mean     std     mean     std          mean     std
modelo                                                                                         
Nulo     0.2654  0.0028  0.6061  0.0300   0.7460  0.0155  -0.4858  0.0091        0.0000  0.0000
Poisson  0.2406  0.0030  0.5895  0.0321   0.6245  0.0166  -0.4250  0.0095        0.1629  0.0077
BN       0.2406  0.0030  0.5895  0.0321   0.6245  0.0166  -0.4250  0.0095        0.1629  0.0077
ZINB     0.2483  0.0029  0.5887  0.0317   0.6179  0.0158  -0.3818  0.0060        0.1718  0.0070
Ajustes que no convergieron en la validación: 0


## 7. Decisión del modelo final

In [12]:
## El ZINB reemplaza a la BN solo si mejora el BIC, el log-score y la devianza fuera de muestra, y si BN y ZINB convergen en todos los ajustes
piv = res_cv.pivot(index='fold', columns='modelo', values=['logscore', 'devianza'])
d_ls = float((piv['logscore']['ZINB'] - piv['logscore']['BN']).mean())
d_dv = float((piv['devianza']['BN'] - piv['devianza']['ZINB']).mean())
crit_bic = aj['ZINB']['bic'] < aj['BN']['bic']
crit_cv = d_ls > 0 and d_dv > 0
conv_ok = all(aj[n]['convergio'] for n in ('BN', 'ZINB')) and bool(res_cv.loc[res_cv['modelo'].isin(['BN', 'ZINB']), 'convergio'].all())

MODELO_FINAL = 'ZINB' if (BASE == 'BN' and crit_bic and crit_cv and conv_ok) else BASE

print(f"BIC BN / ZINB: {aj['BN']['bic']:.1f} / {aj['ZINB']['bic']:.1f} -> favorece ZINB: {crit_bic}")
print(f"CV: mejora media de logscore = {d_ls:+.5f} | de devianza = {d_dv:+.5f} -> favorece ZINB: {crit_cv}")
print(f"BN y ZINB convergieron en todos los ajustes: {conv_ok}")
print(f"Modelo final: {MODELO_FINAL}")

BIC BN / ZINB: 112003.2 / 100661.9 -> favorece ZINB: True
CV: mejora media de logscore = +0.04317 | de devianza = +0.00664 -> favorece ZINB: True
BN y ZINB convergieron en todos los ajustes: True
Modelo final: Poisson


In [13]:
## Coeficientes del modelo final ajustado con todos los segmentos: exp(coef) es IRR en el conteo y OR en la inflación
res_f = aj[MODELO_FINAL]['res']
tab = pd.DataFrame({'coef': _num(res_f.params), 'ee': _num(res_f.bse), 'p': _num(res_f.pvalues)}, index=res_f.params.index)
tab['exp(coef)'] = np.where(tab.index == 'alpha', np.nan, np.exp(tab['coef']))
print(tab.round(4).to_string())

                            coef      ee       p  exp(coef)
const                    -6.8201  0.0208  0.0000     0.0011
dist_puente_peatonal_m    0.0155  0.0064  0.0153     1.0156
dist_puente_vehicular_m   0.1124  0.0068  0.0000     1.1190
dist_semaforo_m          -0.1871  0.0061  0.0000     0.8294
ratio_cobertura_peatonal  0.7401  0.0100  0.0000     2.0962
MVINUMC                   0.1625  0.0092  0.0000     1.1764
vel_60                    0.3134  0.0165  0.0000     1.3681


# Resultados

## 1. Datos y variable respuesta

El conjunto de estimación contiene **131.704 segmentos viales**. La variable respuesta es el conteo de siniestros peatonales por segmento (`n_siniestros`), con la longitud del segmento como offset.

| Siniestros por segmento | Segmentos |
|---|---|
| 0 | 118.382 (89,88 %) |
| 1 | 9.826 |
| 2 | 2.062 |
| 3 | 729 |
| 4 | 332 |
| 5 | 147 |
| 6 | 85 |
| 7 | 41 |
| ≥ 8 | 100 |

Solo el 10,12 % de los segmentos (13.322) registra al menos un siniestro peatonal.

## 2. Selección de la distribución

Ajuste con todos los segmentos:

| Modelo | Parámetros | logLik | AIC | BIC | α | π medio |
|---|---|---|---|---|---|---|
| Nulo | 1 | −63.974,4 | 127.950,7 | 127.960,5 | – | – |
| Poisson | 7 | −55.954,5 | 111.923,0 | 111.991,5 | – | – |
| Binomial Negativa | 8 | −55.954,5 | 111.924,9 | 112.003,2 | ≈ 0 | – |
| ZINB | 10 | −50.272,0 | 100.564,0 | 100.661,9 | 2,253 | 0,133 |

- **Poisson frente a Binomial Negativa.** El parámetro de dispersión de la BN colapsa a α ≈ 0. La razón de verosimilitud es LR = 0,04, con p = 0,42 (corregido por frontera). No hay evidencia de sobredispersión una vez incluidas las covariables, y la BN es indistinguible de Poisson. Por parsimonia se conserva **Poisson** como modelo base.
- **ZINB.** Tiene menor BIC (diferencia de ≈ 11.300 puntos frente a BN) y mejores métricas fuera de muestra (sección 3). La regla de decisión del cuaderno solo permite que el ZINB reemplace a la BN cuando esta es el modelo base. Como la base es Poisson, el modelo final es **Poisson**. La inflación del ZINB depende únicamente de la longitud del segmento, que ya entra como offset.

## 3. Desempeño fuera de muestra (validación cruzada aleatoria, K = 5)

Media y desviación estándar entre pliegues. Ningún ajuste dejó de converger.

| Modelo | MAE | RMSE | Devianza | Log-score | Devianza explicada |
|---|---|---|---|---|---|
| Nulo | 0,2654 (0,0028) | 0,6061 (0,0300) | 0,7460 (0,0155) | −0,4858 (0,0091) | – |
| Poisson = BN | 0,2406 (0,0030) | 0,5895 (0,0321) | 0,6245 (0,0166) | −0,4250 (0,0095) | 16,3 % (0,8) |
| ZINB | 0,2483 (0,0029) | 0,5887 (0,0317) | 0,6179 (0,0158) | −0,3818 (0,0060) | 17,2 % (0,7) |

- Frente al modelo nulo, Poisson reduce el MAE en 9,3 % y el RMSE en 2,7 %.
- Las diferencias de RMSE entre modelos (≈ 0,001) son mucho menores que la desviación entre pliegues (≈ 0,03), por lo que esa métrica no discrimina entre ellos.
- El ZINB mejora la devianza y el log-score, pero no el MAE, que es menor en Poisson.
- Al ser una validación aleatoria, y dado que los siniestros se concentran en corredores contiguos, estas cifras pueden ser optimistas respecto a la generalización espacial.

## 4. Coeficientes del modelo final (Poisson)

Las tres distancias y la cobertura peatonal entran como log(1 + x) estandarizado, por lo que su IRR corresponde a un aumento de **una desviación estándar**. El número de calzadas se interpreta por calzada adicional y la velocidad como 60 km/h frente a la categoría de referencia. El intercepto no se interpreta.

| Variable | Coef. | EE | IRR | IC 95 % | p | Cambio en la frecuencia esperada |
|---|---|---|---|---|---|---|
| Distancia a puente peatonal | 0,0155 | 0,0064 | 1,016 | 1,003 – 1,028 | 0,015 | +1,6 % |
| Distancia a puente vehicular | 0,1124 | 0,0068 | 1,119 | 1,104 – 1,134 | < 0,001 | +11,9 % |
| Distancia a semáforo | −0,1871 | 0,0061 | 0,829 | 0,820 – 0,839 | < 0,001 | −17,1 % |
| Cobertura peatonal | 0,7401 | 0,0100 | 2,096 | 2,055 – 2,138 | < 0,001 | +109,6 % |
| Número de calzadas | 0,1625 | 0,0092 | 1,176 | 1,155 – 1,198 | < 0,001 | +17,6 % |
| Velocidad 60 km/h | 0,3134 | 0,0165 | 1,368 | 1,325 – 1,413 | < 0,001 | +36,8 % |

## 5. Lectura de los resultados

- **Diseño vial.** A igual longitud y demás variables, la frecuencia esperada aumenta con el número de calzadas y con la velocidad reglamentaria de 60 km/h. Ambas asociaciones van en el sentido planteado en H2.
- **Semáforos.** Los segmentos más alejados de un semáforo presentan menor frecuencia esperada, en el sentido esperado.
- **Cobertura peatonal.** Su IRR es ≈ 2,1 y tiene signo contrario al esperado. No debe leerse como que más infraestructura peatonal cause más siniestros. Lo más plausible es que actúe como aproximación a la actividad peatonal, ya que el modelo no incluye un término de exposición peatonal medido. El resultado se mantuvo estable al simplificar la especificación de la velocidad (IRR 2,08 en la versión previa).
- **Puente peatonal.** El efecto es estadísticamente significativo por el tamaño de la muestra, pero prácticamente nulo (IRR 1,016).
- **Velocidad.** Una parte de los valores de `MVI_VELREG` fue imputada con KNN, por lo que el IRR de 60 km/h debe interpretarse con cautela.

## 6. Limitaciones y pendientes

- Los errores estándar asumen independencia entre segmentos y probablemente están subestimados por autocorrelación espacial. Se recomienda recalcularlos con errores robustos.
- No se calculó el índice de Moran sobre los residuales, ni se implementó una partición espacial por bloques.
- `poblacion_expuesta` no está incluida en el modelo; incorporarla podría modificar el coeficiente de cobertura peatonal.
- Falta la aplicación del modelo a toda la red: frecuencia esperada por segmento, ranking, porcentaje del top-N sin siniestros históricos y mapa de priorización.


In [15]:
## Diagnóstico 1: dispersión de Pearson del Poisson y perfil de verosimilitud de alpha con GLM
X, y, off = D['X_tr'], D['y_tr'], D['off_tr']
pois = sm.GLM(y, X, family=sm.families.Poisson(), offset=off).fit()
print(f"Pearson / gl del Poisson: {pois.pearson_chi2 / pois.df_resid:.3f}")  # cerca de 1 = sin sobredispersión

grid = np.r_[1e-3, np.linspace(0.05, 4, 80)]
# El GLM binomial negativo con alpha fijo no depende del optimizador de NegativeBinomial
ll = np.array([sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=a), offset=off).fit().llf for a in grid])
print(f"alpha del perfil: {grid[ll.argmax()]:.3f} | logLik Poisson: {pois.llf:.1f} | logLik máx del perfil: {ll.max():.1f}")

## Diagnóstico 2: elasticidad con la longitud (el offset supone 1)
X_len = X.copy()
X_len['log_long'] = off
p_len = sm.GLM(y, X_len, family=sm.families.Poisson(), offset=off).fit(cov_type='HC0')
b, ci = p_len.params['log_long'], p_len.conf_int().loc['log_long']
# El coeficiente de log_long mide la desviación respecto de 1, por eso se suma 1
print(f"Elasticidad con la longitud: {1 + b:.3f} (IC95 % {1 + ci[0]:.3f} a {1 + ci[1]:.3f})")
print(pd.DataFrame({'IRR offset fijo': np.exp(pois.params), 'IRR longitud libre': np.exp(p_len.params.drop('log_long'))}).round(3).to_string())

## Diagnóstico 3: construcción de la cobertura peatonal en valores crudos
cob = df_unificado['ratio_cobertura_peatonal']
print(cob.describe(percentiles=[.5, .9, .99]).round(3).to_string())
print(f"% de segmentos con cobertura > 1: {100 * (cob > 1).mean():.2f}")
print(f"Spearman cobertura vs longitud: {cob.corr(df_unificado['long_seg_m'], method='spearman'):.3f} | vs n_siniestros: {cob.corr(df_unificado['n_siniestros'], method='spearman'):.3f}")

## Diagnóstico 4: errores robustos (HC0) y concentración de siniestros en el 0,1 % de segmentos con más conteos
rob = sm.GLM(y, X, family=sm.families.Poisson(), offset=off).fit(cov_type='HC0')
print(pd.DataFrame({'EE Poisson': pois.bse, 'EE robusto': rob.bse, 'p robusto': rob.pvalues}).round(4).to_string())
orden = np.sort(y)[::-1]
print(f"% de siniestros en el 0,1 % de segmentos con más conteos: {100 * orden[:int(len(y) * 0.001)].sum() / y.sum():.1f}")

Pearson / gl del Poisson: 2.182
alpha del perfil: 3.150 | logLik Poisson: -55954.5 | logLik máx del perfil: -50374.5
Elasticidad con la longitud: 0.488 (IC95 % 0.455 a 0.521)
                          IRR offset fijo  IRR longitud libre
const                               0.001               0.009
dist_puente_peatonal_m              1.016               0.979
dist_puente_vehicular_m             1.119               1.089
dist_semaforo_m                     0.829               0.793
ratio_cobertura_peatonal            2.096               2.012
MVINUMC                             1.176               1.192
vel_60                              1.368               1.440
count    131704.000
mean          0.410
std           0.454
min           0.000
50%           0.167
90%           1.000
99%           1.000
max           1.000
% de segmentos con cobertura > 1: 0.00
Spearman cobertura vs longitud: 0.094 | vs n_siniestros: 0.238
                          EE Poisson  EE robusto  p robusto
const  

In [16]:
## Refit 1: BN con la longitud como covariable libre (sin offset), arrancando desde el alpha del perfil
X_len = X.copy()
X_len['log_long'] = off
p0 = sm.GLM(y, X_len, family=sm.families.Poisson()).fit()
# El orden de params de NegativeBinomial es [coeficientes, alpha]
nb = sm.NegativeBinomial(y, X_len, loglike_method='nb2').fit(start_params=np.r_[p0.params, 3.15], method='bfgs', maxiter=500, disp=0)
a_hat = nb.params.iloc[-1]
print(f"alpha = {a_hat:.3f} | logLik BN = {nb.llf:.1f} | convergió: {nb.mle_retvals['converged']}")

## Chequeo: con ese alpha el Pearson/gl debe quedar cerca de 1
chk = sm.GLM(y, X_len, family=sm.families.NegativeBinomial(alpha=a_hat)).fit()
print(f"Pearson / gl con alpha estimado: {chk.pearson_chi2 / chk.df_resid:.3f}")

## Refit 2: cobertura por tramos en lugar del ratio continuo, para ver la relación dosis-respuesta y el efecto de la saturación
cob = df_unificado['ratio_cobertura_peatonal']
tramos = pd.cut(cob, [-0.001, 0.05, 0.25, 0.5, 0.999, 1.0], labels=['0-5%', '5-25%', '25-50%', '50-99%', '100%'])
# drop_first deja 0-5 % como referencia
T = pd.get_dummies(tramos, prefix='cob', drop_first=True, dtype=float)
X_t = pd.concat([X_len.drop(columns='ratio_cobertura_peatonal'), T], axis=1)
m_t = sm.GLM(y, X_t, family=sm.families.NegativeBinomial(alpha=a_hat)).fit(cov_type='HC0')
print(pd.DataFrame({'IRR': np.exp(m_t.params), 'p': m_t.pvalues}).filter(like='cob_', axis=0).round(3).to_string())
print(tramos.value_counts().sort_index().to_string())

alpha = 2.830 | logLik BN = -49735.5 | convergió: True
Pearson / gl con alpha estimado: 1.164
              IRR    p
cob_5-25%   1.914  0.0
cob_25-50%  2.135  0.0
cob_50-99%  2.869  0.0
cob_100%    5.210  0.0
ratio_cobertura_peatonal
0-5%      57729
5-25%     15624
25-50%     9944
50-99%     1845
100%      46562
